# RAG Retrieval Basics - 完全版
上から順にセルを実行してください。

In [1]:
# ① プロジェクトフォルダ作成

from pathlib import Path

ROOT = Path("/Users/mh/Downloads/Mini Project/Mar2026/rag_retrieval_basics")

for d in [
    ROOT / "data" / "docs",
    ROOT / "data" / "queries",
    ROOT / "runs",
]:
    d.mkdir(parents=True, exist_ok=True)

registry = ROOT / "runs" / "registry.csv"

if not registry.exists():
    registry.write_text(
        "run_id,run_name,timestamp,model_type,n_docs,n_queries,recall_at_1,recall_at_3,mrr\n",
        encoding="utf-8"
    )

print("Project root:", ROOT)

Project root: /Users/mh/Downloads/Mini Project/Mar2026/rag_retrieval_basics


In [2]:
# ② 技術ドキュメント作成

DOC_DIR = ROOT / "data" / "docs"

docs = {
    "doc_01_tls.txt": """TLS handshake starts with ClientHello. The server replies with ServerHello and sends its certificate. The client validates the certificate before establishing the session key.""",
    "doc_02_apache.txt": """To enable HTTPS in Apache you need mod_ssl, a server certificate, a private key, and a VirtualHost on port 443. The SSLEngine directive must be on.""",
    "doc_03_nginx.txt": """In Nginx HTTPS is configured inside a server block with listen 443 ssl. You must specify ssl_certificate and ssl_certificate_key.""",
    "doc_04_dns.txt": """DNS resolves hostnames into IP addresses. Recursive resolvers query authoritative name servers to return the final answer.""",
    "doc_05_scep.txt": """SCEP is used for certificate enrollment. A device sends a CSR to the CA or RA and receives a certificate.""",
    "doc_06_ocsp.txt": """OCSP checks the revocation status of a certificate in real time. It is an alternative to downloading a full CRL.""",
    "doc_07_rootca.txt": """A root CA is the trust anchor in a PKI hierarchy. Intermediate CAs chain back to the trusted root certificate.""",
    "doc_08_intune.txt": """Microsoft Intune can deploy certificate profiles to devices. SCEP certificate profiles are commonly used for automatic enrollment.""",
}

for name, text in docs.items():
    (DOC_DIR / name).write_text(text, encoding="utf-8")

print("Documents created:", len(docs))

Documents created: 8


In [3]:
# ③ クエリ作成

import json

QUERY_PATH = ROOT / "data" / "queries" / "queries.jsonl"

queries_data = [
    {"qid": "q001", "query": "How does TLS verify certificates?", "gold_doc": "doc_01_tls.txt"},
    {"qid": "q002", "query": "How do you enable HTTPS on Apache?", "gold_doc": "doc_02_apache.txt"},
    {"qid": "q003", "query": "How is HTTPS configured in Nginx?", "gold_doc": "doc_03_nginx.txt"},
    {"qid": "q004", "query": "What does DNS do?", "gold_doc": "doc_04_dns.txt"},
    {"qid": "q005", "query": "Which protocol enrolls certificates using CSR?", "gold_doc": "doc_05_scep.txt"},
    {"qid": "q006", "query": "How can certificate revocation be checked without CRL?", "gold_doc": "doc_06_ocsp.txt"},
    {"qid": "q007", "query": "What is the trust anchor in PKI?", "gold_doc": "doc_07_rootca.txt"},
    {"qid": "q008", "query": "Which Microsoft service deploys SCEP profiles?", "gold_doc": "doc_08_intune.txt"},
]

with QUERY_PATH.open("w", encoding="utf-8") as f:
    for r in queries_data:
        f.write(json.dumps(r) + "\n")

print("Queries created:", len(queries_data))

Queries created: 8


In [4]:
# ④ Retrieval（TF-IDF）

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_files = sorted(DOC_DIR.glob("*.txt"))
doc_names = [p.name for p in doc_files]
doc_texts = [p.read_text(encoding="utf-8") for p in doc_files]

queries = []
with QUERY_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        queries.append(json.loads(line))

vectorizer = TfidfVectorizer()
doc_matrix = vectorizer.fit_transform(doc_texts)

print("Docs:", len(doc_texts), "Queries:", len(queries))

Docs: 8 Queries: 8


In [5]:
type(vectorizer)

sklearn.feature_extraction.text.TfidfVectorizer

In [6]:
# ⑤ Retrieval評価

recall1 = 0
recall3 = 0
rr_sum = 0.0

for q in queries:
    q_vec = vectorizer.transform([q["query"]])
    sims = cosine_similarity(q_vec, doc_matrix)[0]
    ranked_idx = np.argsort(sims)[::-1]
    ranked_docs = [doc_names[i] for i in ranked_idx]

    if ranked_docs[0] == q["gold_doc"]:
        recall1 += 1

    if q["gold_doc"] in ranked_docs[:3]:
        recall3 += 1

    if q["gold_doc"] in ranked_docs:
        rank = ranked_docs.index(q["gold_doc"]) + 1
        rr_sum += 1.0 / rank

recall_at_1 = recall1 / len(queries)
recall_at_3 = recall3 / len(queries)
mrr = rr_sum / len(queries)

print("Recall@1:", recall_at_1)
print("Recall@3:", recall_at_3)
print("MRR:", mrr)

Recall@1: 1.0
Recall@3: 1.0
MRR: 1.0


In [7]:
recall_at_1

1.0

In [8]:
# ⑥ LLMロード（初回はモデルダウンロードに数分かかります）

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

LLM_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading LLM:", LLM_MODEL)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if torch.backends.mps.is_available() else torch.float32
)

device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)
model.eval()

print("LLM device:", device)

Loading LLM: TinyLlama/TinyLlama-1.1B-Chat-v1.0


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LLM device: mps


In [9]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [10]:
# ⑦ RAGプロンプト

RAG_PROMPT = """You are a technical assistant.

Use the CONTEXT to answer the QUESTION.

If the answer is not in the context say:
"I don't know based on the documents."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

In [11]:
# ⑧ LLM回答関数

def generate_answer(question, context, max_new_tokens=40):
    prompt = RAG_PROMPT.format(
        context=context,
        question=question
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    if "ANSWER:" in text:
        text = text.split("ANSWER:")[-1].strip()
    
    del inputs, output          # 追加：不要な変数を削除
    torch.mps.empty_cache()     # 追加：GPUメモリを解放
    
    return text

print("generate_answer function defined.")

generate_answer function defined.


In [12]:
type(generate_answer)

function

In [ ]:
# ⑨ Retrieval + Generation（RAG）

results = []

for q in queries:
    query = q["query"]

    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, doc_matrix)[0]

    ranked_idx = np.argsort(sims)[::-1]
    top_idx = ranked_idx[:3]

    retrieved_docs = [doc_names[i] for i in top_idx]
    retrieved_text = "\n".join([doc_texts[i] for i in top_idx])

    answer = generate_answer(query, retrieved_text)

    results.append({
        "qid": q["qid"],
        "query": query,
        "gold_doc": q["gold_doc"],
        "retrieved_docs": ";".join(retrieved_docs),
        "answer": answer
    })

df_rag = pd.DataFrame(results)
df_rag.head()

Both `max_new_tokens` (=40) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
# ⑩ Run保存

import datetime

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = "rag_tinyllama_v1"
run_id = f"{ts}__{run_name}"

RUN_DIR = ROOT / "runs" / run_id
RUN_DIR.mkdir(parents=True, exist_ok=True)

df_rag.to_csv(
    RUN_DIR / "rag_answers.tsv",
    sep="\t",
    index=False,
    encoding="utf-8"
)

print("Run saved:", RUN_DIR)

In [ ]:
DOC_DIR

SyntaxError: invalid syntax (3514729975.py, line 1)